# Theta-gamma coupling states in REM — PFC vs HPC (**v2: Butterworth + Hilbert**)

End-to-end re-implementation of the analyses in **Zhang et al. 2019,
eLife, "Sub-second dynamics of theta-gamma coupling in hippocampal
CA1"**, adapted to our PFC + HPC RGS data and stratified by **phasic vs
tonic REM**.

## What's different from v1

This is **v2**. The only change vs v1 is the theta cycle detector:

| | v1 (`zhang_tg_states_pfc_hpc.ipynb`) | **v2 (this notebook)** |
|---|---|---|
| theta cycle method | mask-sift **EMD** on HPC, theta-IMF Hilbert phase | **Butterworth bandpass 5-12 Hz → Hilbert → 2π crossings** (Zhang's original method) |
| `is_good`, duration & amplitude filters | yes | no — only monotonicity + 5-12 Hz instantaneous-frequency window |
| per-sample phase reference | `np.angle(hilbert(theta_IMF))` | `np.unwrap(np.angle(hilbert(bandpass(LFP, 5-12 Hz))))` |
| pipeline driver | `process_rat_emd` | **`process_rat_butter`** |
| analysis fs | 1000 Hz (native) | **625 Hz** (Zhang's analysis_fs) |

Everything downstream — Morlet wavelet FPP construction (smoothing,
z-score, 20 phase bins), k-means clustering, gravity-based S/M/EF/LF
labelling, intra/inter correlation, Markov transitions, PFC-HPC PPC,
and all plots — is the **same code** as v1, because all those modules
consume a `RatAggregate` and don't care which cycle detector produced
the cycles.

## How v2 handles phasic vs tonic without dropping short intervals

Whereas the old `process_rat` sliced the LFP per phasic/tonic interval
and ran Butterworth on each slice (which dropped short phasic
intervals because of the 2 s minimum-length guard), **v2 runs the
Butterworth + Hilbert pipeline once on the FULL session LFP** and
tags each cycle by which substate interval fully contains it. So
short phasic intervals contribute every cycle they contain, just
like v1 does for EMD cycles.

## Method (full recap)

1. **Wavelet spectrum normalized by theta phase** — Morlet CWT on HPC
   LFP at 625 Hz, sequential time/frequency boxcar smoothing (±8 ms ×
   ±2 Hz, matching Zhang's `ntw = 11`, `nsw = 3`), z-score per
   frequency, then 20 equal theta-phase bins → **FPP** = 79 × 20
   matrix per cycle.
2. **Theta cycle detector**: Butterworth 4th-order bandpass 5-12 Hz
   (zero-phase `sosfiltfilt`), `np.angle(hilbert(.))`, `np.unwrap(.)`,
   then 2π crossings. Cycles whose instantaneous frequency (= 1/dur)
   falls outside 5-12 Hz are dropped. **This is exactly the method
   from Zhang's Materials and Methods, "Wavelet spectrum normalized
   by theta phase".**
3. **k-means clustering** with **Pearson-correlation distance**
   (`D = 1 − r`) and **k = 4**. Cluster sorting by gravity frequency
   + gravity phase per `PhaseFreSort.m`.
4. **Intra- vs inter-cluster correlation** under 5-fold CV.
5. **Cross-rat assignment accuracy**.
6. **Markov state transitions** (4 × 4) and state occurrences.
7. **LFP-LFP PPC** — wavelet cross-spectrum (PFC × HPC*), V-statistic.

Project-specific adaptations:

* **CA3-CA1 and EC-CA1** in Zhang are replaced by **PFC-HPC**.
* **Pre / maze / post-track** in Zhang is replaced by **phasic vs
  tonic REM** (`extract_pt_intervals` in `src/utils.py`).
* All PPC heatmaps use the **hot** colormap.
* Convention: `phase_lag = angle(PFC * conj(HPC))`. Positive lag →
  PFC leads HPC at that (frequency, theta phase).

The pipeline modules live in `exploration/zhang_tg_states/`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make project modules importable
HERE = os.getcwd()
for rel in ('.', '..', '../..'):
    cand = os.path.abspath(os.path.join(HERE, rel))
    if cand not in sys.path:
        sys.path.insert(0, cand)
for rel in ('exploration', '../exploration'):
    cand = os.path.abspath(os.path.join(HERE, rel))
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

from zhang_tg_states import (
    ANALYSIS_FS, FREQUENCIES, N_PHASE_BINS,
    PHASE_CENTERS_DEG, PHASE_CENTERS_RAD,
    STATE_NAMES,
    cluster_fpps_into_tg_states,
    fpps_from_lfp_segment,
    intra_inter_correlation, kfold_intra_inter,
    cross_dataset_accuracy, pairwise_cross_dataset,
    phasic_vs_tonic_transitions,
    cross_spectrum_for_segment, state_conditioned_ppc,
    ppc_phase_pooled,
)
from zhang_tg_states import plotting as zplot
from zhang_tg_states.data_pipeline import (
    BASE_PATH, RAT_GROUPS, process_rat_butter,
)

print('frequencies:', FREQUENCIES[0], '...', FREQUENCIES[-1],
      f'({len(FREQUENCIES)} bins)')
print('phase bins:', N_PHASE_BINS, '|', 'analysis fs:', ANALYSIS_FS, 'Hz')
print('state names:', STATE_NAMES)
print('RAT_GROUPS:', RAT_GROUPS)


## Optional — load previously-saved results

If you have already run this notebook once and saved the pickles in
section 7, set `LOAD_FROM_PICKLE = True` below and run only this cell;
then **skip down to section 8** ("FPP × transitions") at the end. The
load reconstructs `per_rat_cluster`, `trans_per_rat`, and `per_rat_ppc`
so the analyses at the end and any of the plot cells that consume
only those dicts will work without rerunning the slow data ingestion
and clustering.

If `LOAD_FROM_PICKLE = False` (default), the notebook runs the full
pipeline below normally.


In [ ]:
LOAD_FROM_PICKLE = False     # flip to True after section 7 has saved pickles

if LOAD_FROM_PICKLE:
    from types import SimpleNamespace
    out_dir_load = os.path.join(HERE, 'zhang_tg_states')
    per_rat_cluster = {'positive': {}, 'control': {}}
    trans_per_rat   = {'positive': {}, 'control': {}}
    per_rat_ppc     = {'positive': {}, 'control': {}}
    for group in ['positive', 'control']:
        fname = os.path.join(out_dir_load, f'tg_states_v2_{group}_pfc_hpc.pkl')
        if not os.path.exists(fname):
            print(f'  [load] missing {fname} - skipping {group}')
            continue
        with open(fname, 'rb') as fh:
            payload = pickle.load(fh)
        for rid, cl_dict in payload['clusters'].items():
            per_rat_cluster[group][rid] = SimpleNamespace(**cl_dict)
        for rid, tt_dict in payload.get('transitions', {}).items():
            trans_per_rat[group][rid] = {
                k: SimpleNamespace(**v) for k, v in tt_dict.items()
            }
        for rid, ppc_dict in payload.get('ppc', {}).items():
            per_rat_ppc[group][rid] = {
                k: (None if v is None else SimpleNamespace(**v))
                for k, v in ppc_dict.items()
            }
        print(f'  [load] {group}: loaded {len(payload["clusters"])} rats')

    # raw cycles are NOT in the pickle, so any cell that uses
    # results[group][rid].fpps.fpps or .cross_spectrum.angles will fail.
    # That includes: data ingestion, intra/inter, cross-rat accuracy,
    # PPC re-computation, and the per-rat phasic-vs-tonic m-FPP plot
    # cell (it uses raw cycles to compute substate m-FPPs).
    # Substate m-FPPs were also saved in the pickle (`m_fpps_phasic`
    # and `m_fpps_tonic` attributes on each per_rat_cluster value), so
    # the group-average phasic-vs-tonic m-FPP cell AND the section-10
    # analysis still work.
    results = None
    print('Loaded. Skip sections 1, 3, 4, and the data-needed parts of '
          'the m-FPP visualisations; sections 5, 6, 8, 9, 10 work with '
          'the loaded data.')
else:
    print('Full pipeline will run below. After section 7 saves pickles, '
          'rerun this cell with LOAD_FROM_PICKLE=True to skip the slow '
          'cells next time.')


## 1 — Per-rat data ingestion (Butterworth + Hilbert theta cycles)

`process_rat_butter(rat_id)` walks every condition folder under
`BASE_PATH`, loads HPC + PFC LFPs and the hypnogram for each session,
and runs **Zhang's original cycle-detection pipeline** on the full
continuous session LFP:

  1. Resample HPC and PFC to **625 Hz** (`analysis_fs`).
  2. 4th-order **Butterworth bandpass HPC at 5-12 Hz** (zero-phase
     `sosfiltfilt`).
  3. `np.angle(hilbert(.))` → wrapped phase → `np.unwrap(.)` → monotonic
     theta phase φ(t).
  4. Detect cycles as samples where φ crosses successive multiples
     of 2π. Reject cycles whose instantaneous frequency (= 1/duration)
     is outside 5-12 Hz.
  5. Compute the **Morlet wavelet FPP** per cycle.
  6. Compute the **PFC × HPC* cross-spectrum** per cycle.
  7. `extract_pt_intervals` → phasic and tonic REM IntervalSets.
  8. Tag each cycle by which phasic/tonic interval fully contains it;
     drop cycles in wake/NREM.
  9. Assign a unique `segment_id` per (session, substate, interval) so
     the Markov module can group runs.

**No EMD anywhere.** This is the most faithful re-implementation of
Zhang's original method on this dataset.

Adjust `RUN_RATS` to limit which rats are processed during development.


In [ ]:
# Toggle: which rats to process
RUN_RATS = {
    'positive': RAT_GROUPS['positive'],   # [3, 4, 7, 8]
    'control':  RAT_GROUPS['control'],    # [1, 2, 6, 9]
}

results = {'positive': {}, 'control': {}}
for group_name, rat_ids in RUN_RATS.items():
    print(f'\n=== group: {group_name} ===')
    for rid in rat_ids:
        agg = process_rat_butter(rid, verbose=True)
        if agg is None:
            continue
        results[group_name][rid] = agg

n_total = sum(len(v) for v in results.values())
print(f'\nLoaded {n_total} rat aggregates')
for g, d in results.items():
    print(f'  {g}: rats = {sorted(d.keys())}')


### Quick summary table

How many cycles per rat per substate?


In [ ]:
rows = []
for group, rats in results.items():
    for rid, agg in rats.items():
        n_phasic = int((agg.cycle_substates == 'phasic').sum())
        n_tonic  = int((agg.cycle_substates == 'tonic').sum())
        rows.append(dict(group=group, rat=rid,
                         n_cycles_phasic=n_phasic,
                         n_cycles_tonic=n_tonic,
                         n_sessions=len(agg.sessions),
                         total_cycles=len(agg.cycle_substates)))
cycle_summary = pd.DataFrame(rows).sort_values(['group','rat']).reset_index(drop=True)
cycle_summary


### Diagnostic: Zhang-style theta cycles on HPC and the matching PFC chunks

Before clustering, let's visualise the cycle detector at work. For one
rat / substate / REM bout we plot:

* **Top panel** — HPC raw LFP + HPC bandpass (5-12 Hz, Butterworth
  order 4, `sosfiltfilt`) + the cycle starts (green ●) and ends
  (red ■) detected by Zhang's method (2π crossings of the unwrapped
  Hilbert phase + 5-12 Hz instantaneous-frequency filter).
* **Bottom panel** — PFC raw LFP + PFC bandpass in the same band +
  the **same HPC cycle markers overlaid**. This shows the time chunks
  of PFC that go into each cycle's PFC × HPC* cross-spectrum used in
  the PPC analysis.

Change `INSPECT_RAT`, `INSPECT_SUBSTATE`, `INSPECT_DURATION_SEC` to
inspect a different period.


In [ ]:
from scipy.signal import butter, hilbert, sosfiltfilt
from zhang_tg_states.fpp import (
    theta_unwrapped_phase, extract_theta_cycles, resample_to_analysis_fs,
    THETA_BAND, THETA_CYCLE_FREQ_RANGE,
)
from zhang_tg_states.data_pipeline import list_rat_sessions, _intervalset_to_arrays
sys.path.insert(0, os.path.join(HERE, '..', 'src'))
sys.path.insert(0, os.path.join(HERE, 'src'))
from utils import get_data, extract_pt_intervals

INSPECT_RAT = 3
INSPECT_SUBSTATE = 'tonic'          # 'phasic' or 'tonic'
INSPECT_DURATION_SEC = 5.0

sessions_to_search = list_rat_sessions(INSPECT_RAT)
bundle = None
for sess in sessions_to_search:
    try:
        hpc, hypno, fs = get_data(sess['hpc_path'], sess['state_path'], type='hpc')
        pfc, _, _ = get_data(sess['pfc_path'], sess['state_path'], type='pfc')
    except Exception:
        continue
    L = min(len(hpc), len(pfc))
    hpc = np.asarray(hpc[:L], dtype=float)
    pfc = np.asarray(pfc[:L], dtype=float)
    fs = int(fs)
    try:
        phasic_iv, tonic_iv, _ = extract_pt_intervals(hpc, hypno, fs=fs)
    except Exception:
        continue
    iv = phasic_iv if INSPECT_SUBSTATE == 'phasic' else tonic_iv
    if iv is None or len(iv) == 0:
        continue
    starts, ends = _intervalset_to_arrays(iv)
    durs = ends - starts
    if not np.any(durs >= INSPECT_DURATION_SEC):
        continue
    idx = int(np.argmax(durs))
    start_sec = float(starts[idx])
    end_sec = min(start_sec + INSPECT_DURATION_SEC, float(ends[idx]))
    bundle = dict(sess=sess, hpc=hpc, pfc=pfc, fs=fs,
                  start_sec=start_sec, end_sec=end_sec, idx=idx)
    break

if bundle is None:
    print(f'no usable {INSPECT_SUBSTATE} interval >= {INSPECT_DURATION_SEC}s for rat {INSPECT_RAT}')
else:
    sess, hpc, pfc, fs = bundle['sess'], bundle['hpc'], bundle['pfc'], bundle['fs']
    start_sec, end_sec, idx = bundle['start_sec'], bundle['end_sec'], bundle['idx']
    #print(f'session: {sess[\"folder\"]}/{sess[\"sub\"]}')
    print(f'{INSPECT_SUBSTATE} interval #{idx}: {start_sec:.2f}-{end_sec:.2f}s')

    s_idx = int(round(start_sec * fs))
    e_idx = int(round(end_sec * fs))
    hpc_chunk = hpc[s_idx:e_idx]
    pfc_chunk = pfc[s_idx:e_idx]

    # Resample to analysis_fs (625 Hz) — Zhang's protocol
    hpc_rs, fs_a = resample_to_analysis_fs(hpc_chunk, fs)
    pfc_rs, _    = resample_to_analysis_fs(pfc_chunk, fs)
    L_rs = min(len(hpc_rs), len(pfc_rs))
    hpc_rs = hpc_rs[:L_rs]; pfc_rs = pfc_rs[:L_rs]

    # Zhang's cycle detection: bandpass HPC -> Hilbert -> unwrap -> 2π crossings
    theta_phase = theta_unwrapped_phase(hpc_rs, fs_a, theta_band=THETA_BAND)
    hpc_cycles = extract_theta_cycles(theta_phase, fs_a,
                                       freq_range=THETA_CYCLE_FREQ_RANGE)
    print(f'detected {len(hpc_cycles)} HPC theta cycles in {INSPECT_DURATION_SEC:.1f}s')

    # Bandpass HPC and PFC at the same band for visualisation
    nyq = 0.5 * fs_a
    sos_theta = butter(4, [THETA_BAND[0]/nyq, THETA_BAND[1]/nyq],
                       btype='bandpass', output='sos')
    hpc_bp = sosfiltfilt(sos_theta, hpc_rs)
    pfc_bp = sosfiltfilt(sos_theta, pfc_rs)

    t = np.arange(L_rs) / fs_a
    fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True,
                              constrained_layout=True)

    # ---- HPC ----
    ax = axes[0]
    ax.plot(t, hpc_rs, color='lightgray', lw=0.7,
             label='HPC raw LFP (resampled 625 Hz)', zorder=1)
    ax.plot(t, hpc_bp, color='tab:red', lw=1.3,
             label=f'HPC bandpass {THETA_BAND[0]}-{THETA_BAND[1]} Hz', zorder=2)
    for s, e in hpc_cycles:
        e_clamp = min(e, L_rs - 1)
        ax.axvline(t[s], color='tab:green', lw=0.3, alpha=0.45)
        ax.axvline(t[e_clamp], color='tab:red', lw=0.3, alpha=0.35)
        ax.plot(t[s], hpc_bp[s], 'o', color='tab:green',
                 markersize=7, markeredgecolor='black',
                 markeredgewidth=0.6, zorder=5)
        ax.plot(t[e_clamp], hpc_bp[e_clamp], 's', color='tab:red',
                 markersize=6, markeredgecolor='black',
                 markeredgewidth=0.6, zorder=5)
    ax.plot([], [], 'o', color='tab:green', markeredgecolor='black',
             markersize=7, label='HPC cycle START')
    ax.plot([], [], 's', color='tab:red', markeredgecolor='black',
             markersize=6, label='HPC cycle END')
    ax.set_title(f'HPC: Zhang cycle detection (Butterworth + Hilbert + 2π crossings)'
                 f'  |  {len(hpc_cycles)} cycles in window')
    ax.set_ylabel('HPC (a.u.)')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.grid(alpha=0.2)

    # ---- PFC with HPC cycle boundaries overlaid ----
    ax = axes[1]
    ax.plot(t, pfc_rs, color='lightgray', lw=0.7,
             label='PFC raw LFP (resampled 625 Hz)', zorder=1)
    ax.plot(t, pfc_bp, color='tab:blue', lw=1.3,
             label=f'PFC bandpass {THETA_BAND[0]}-{THETA_BAND[1]} Hz', zorder=2)
    for s, e in hpc_cycles:
        e_clamp = min(e, L_rs - 1)
        ax.axvline(t[s], color='tab:green', lw=0.3, alpha=0.45)
        ax.axvline(t[e_clamp], color='tab:red', lw=0.3, alpha=0.35)
        ax.plot(t[s], pfc_bp[s], 'o', color='tab:green',
                 markersize=7, markeredgecolor='black',
                 markeredgewidth=0.6, zorder=5)
        ax.plot(t[e_clamp], pfc_bp[e_clamp], 's', color='tab:red',
                 markersize=6, markeredgecolor='black',
                 markeredgewidth=0.6, zorder=5)
    ax.plot([], [], 'o', color='tab:green', markeredgecolor='black',
             markersize=7, label='HPC cycle START (on PFC)')
    ax.plot([], [], 's', color='tab:red', markeredgecolor='black',
             markersize=6, label='HPC cycle END (on PFC)')
    ax.set_title('PFC: same HPC cycle boundaries overlaid '
                 '(PFC is binned by HPC theta phase in every analysis below)')
    ax.set_ylabel('PFC (a.u.)')
    ax.set_xlabel('time within interval (s)')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.grid(alpha=0.2)

    fig.suptitle(f'Zhang-style HPC theta cycles + matched PFC chunks  |  '
                 f'rat {INSPECT_RAT}, {INSPECT_SUBSTATE} interval #{idx}, '
                 f'{start_sec:.1f}-{end_sec:.1f}s ({INSPECT_DURATION_SEC:.1f}s shown)',
                 fontsize=11)
    plt.show()


### Average theta cycle shape (Zhang-style detection: Butterworth + Hilbert + 2π crossings)

In [ ]:

import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, sosfiltfilt

# project imports
for rel in ('exploration', '../exploration', os.path.join('..', 'src'), 'src'):
    cand = os.path.abspath(rel)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

from zhang_tg_states.fpp import (
    THETA_BAND,
    THETA_CYCLE_FREQ_RANGE,
    extract_theta_cycles,
    resample_to_analysis_fs,
    theta_unwrapped_phase,
)
from zhang_tg_states.data_pipeline import (
    _intervalset_to_arrays,
    list_rat_sessions,
)
from utils import extract_pt_intervals, get_data  # from project src/

# ---------------- config ----------------
INSPECT_RAT = 3
N_POINTS = 100         # resample each cycle to this many points before averaging
MIN_INTERVAL_SEC = 2.0 # skip intervals shorter than this (filter needs some samples)

# ---------------- collect cycle shapes from every phasic / tonic interval ----------------
def cycle_shapes_from_chunk(lfp_chunk, fs, n_points=N_POINTS):
    """Run Zhang's cycle detection on one continuous chunk and return:
      raw_shapes  -- (n_cycles, n_points) resampled raw-LFP shapes
      bp_shapes   -- (n_cycles, n_points) resampled bandpassed shapes
    """
    lfp_rs, fs_a = resample_to_analysis_fs(lfp_chunk, fs)   # to 625 Hz
    if len(lfp_rs) < int(2 * fs_a):
        return np.empty((0, n_points)), np.empty((0, n_points))
    phase = theta_unwrapped_phase(lfp_rs, fs_a, theta_band=THETA_BAND)
    cycles = extract_theta_cycles(phase, fs_a, freq_range=THETA_CYCLE_FREQ_RANGE)
    if not cycles:
        return np.empty((0, n_points)), np.empty((0, n_points))

    nyq = 0.5 * fs_a
    sos = butter(4, [THETA_BAND[0] / nyq, THETA_BAND[1] / nyq],
                 btype='bandpass', output='sos')
    lfp_bp = sosfiltfilt(sos, lfp_rs)

    x_new = np.linspace(0.0, 1.0, n_points)
    raw_shapes, bp_shapes = [], []
    for s, e in cycles:
        seg_raw = lfp_rs[s:e]
        seg_bp = lfp_bp[s:e]
        if len(seg_raw) < 4:
            continue
        x_old = np.linspace(0.0, 1.0, len(seg_raw))
        raw_shapes.append(np.interp(x_new, x_old, seg_raw))
        bp_shapes.append(np.interp(x_new, x_old, seg_bp))
    if not raw_shapes:
        return np.empty((0, n_points)), np.empty((0, n_points))
    return np.stack(raw_shapes, 0), np.stack(bp_shapes, 0)


# Find the first usable session for this rat and accumulate shapes across ALL sessions
phasic_raw_all, phasic_bp_all = [], []
tonic_raw_all, tonic_bp_all = [], []
n_sessions_used = 0

for sess in list_rat_sessions(INSPECT_RAT):
    try:
        hpc, hypno, fs = get_data(sess['hpc_path'], sess['state_path'], type='hpc')
    except Exception:
        continue
    hpc = np.asarray(hpc, dtype=float)
    fs = int(fs)
    try:
        phasic_iv, tonic_iv, _ = extract_pt_intervals(hpc, hypno, fs=fs)
    except Exception:
        continue

    p_starts, p_ends = _intervalset_to_arrays(phasic_iv) if len(phasic_iv) else (np.array([]), np.array([]))
    t_starts, t_ends = _intervalset_to_arrays(tonic_iv) if len(tonic_iv) else (np.array([]), np.array([]))
    if len(p_starts) == 0 and len(t_starts) == 0:
        continue

    for starts, ends, raw_acc, bp_acc in [
        (p_starts, p_ends, phasic_raw_all, phasic_bp_all),
        (t_starts, t_ends, tonic_raw_all,  tonic_bp_all),
    ]:
        for s_sec, e_sec in zip(starts, ends):
            if e_sec - s_sec < MIN_INTERVAL_SEC:
                continue
            s_idx = int(round(s_sec * fs))
            e_idx = int(round(e_sec * fs))
            chunk = hpc[s_idx:e_idx]
            r, b = cycle_shapes_from_chunk(chunk, fs)
            if r.size == 0:
                continue
            raw_acc.append(r)
            bp_acc.append(b)
    n_sessions_used += 1

phasic_raw = np.concatenate(phasic_raw_all, 0) if phasic_raw_all else np.empty((0, N_POINTS))
phasic_bp  = np.concatenate(phasic_bp_all,  0) if phasic_bp_all  else np.empty((0, N_POINTS))
tonic_raw  = np.concatenate(tonic_raw_all,  0) if tonic_raw_all  else np.empty((0, N_POINTS))
tonic_bp   = np.concatenate(tonic_bp_all,   0) if tonic_bp_all   else np.empty((0, N_POINTS))

print(f'sessions used: {n_sessions_used}')
print(f'phasic cycles: {len(phasic_raw)}')
print(f'tonic  cycles: {len(tonic_raw)}')

# ---------------- plot ----------------
x_deg = np.linspace(0.0, 360.0, N_POINTS)
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5),
                          sharex=True, constrained_layout=True)
panels = [
    (axes[0, 0], phasic_raw, 'phasic — raw LFP',          'tab:blue'),
    (axes[0, 1], tonic_raw,  'tonic — raw LFP',           'tab:orange'),
    (axes[1, 0], phasic_bp,  'phasic — bandpass 5–12 Hz', 'tab:blue'),
    (axes[1, 1], tonic_bp,   'tonic — bandpass 5–12 Hz',  'tab:orange'),
]
for ax, data, label, color in panels:
    if data.size == 0:
        ax.set_title(f'{label}: no data')
        ax.set_axis_off()
        continue
    mean = np.mean(data, axis=0)
    sd   = np.std(data, axis=0, ddof=1)
    ax.plot(x_deg, mean, color=color, lw=2.0, label='mean')
    ax.fill_between(x_deg, mean - sd, mean + sd,
                    color=color, alpha=0.22, linewidth=0,
                    label='±1 SD')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'{label}  (n = {len(data)} cycles)')
    ax.set_ylabel('amplitude (a.u.)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right', frameon=False)
axes[-1, 0].set_xlabel('phase within cycle (deg, 0 to 360)')
axes[-1, 1].set_xlabel('phase within cycle (deg, 0 to 360)')

fig.suptitle(
    f'Rat {INSPECT_RAT} — average theta cycle shape (Zhang detector)\n'
    f'each cycle resampled to {N_POINTS} points (0–360°), then averaged across cycles. '
    f'shaded = ±1 SD',
    fontsize=11,
)
plt.show()

### Average theta cycle shape (Zhang-style), phase-shifted to put TROUGH at 0

In [ ]:

import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, sosfiltfilt

for rel in ('exploration', '../exploration', os.path.join('..', 'src'), 'src'):
    cand = os.path.abspath(rel)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

from zhang_tg_states.fpp import (
    THETA_BAND,
    THETA_CYCLE_FREQ_RANGE,
    extract_theta_cycles,
    resample_to_analysis_fs,
    theta_unwrapped_phase,
)
from zhang_tg_states.data_pipeline import (
    _intervalset_to_arrays,
    list_rat_sessions,
)
from utils import extract_pt_intervals, get_data  # project src/

# ---------------- config ----------------
INSPECT_RAT = 3
N_POINTS = 100           # number of samples per resampled cycle
MIN_INTERVAL_SEC = 2.0   # skip intervals shorter than this


def trough_aligned_cycle_shapes(lfp_chunk, fs, n_points=N_POINTS):
    """Detect cycles via Zhang, resample each to ``n_points``, then circularly
    roll each cycle so its bandpass-trough sits at index 0 (= 0° on the
    output axis).

    Returns (raw_shapes, bp_shapes) shaped ``(n_cycles, n_points)``.
    """
    lfp_rs, fs_a = resample_to_analysis_fs(lfp_chunk, fs)
    if len(lfp_rs) < int(2 * fs_a):
        return np.empty((0, n_points)), np.empty((0, n_points))
    phase = theta_unwrapped_phase(lfp_rs, fs_a, theta_band=THETA_BAND)
    cycles = extract_theta_cycles(phase, fs_a, freq_range=THETA_CYCLE_FREQ_RANGE)
    if not cycles:
        return np.empty((0, n_points)), np.empty((0, n_points))

    nyq = 0.5 * fs_a
    sos = butter(4, [THETA_BAND[0] / nyq, THETA_BAND[1] / nyq],
                 btype='bandpass', output='sos')
    lfp_bp = sosfiltfilt(sos, lfp_rs)

    x_new = np.linspace(0.0, 1.0, n_points, endpoint=False)
    raw_shapes, bp_shapes = [], []
    for s, e in cycles:
        seg_raw = lfp_rs[s:e]
        seg_bp = lfp_bp[s:e]
        if len(seg_raw) < 4:
            continue
        x_old = np.linspace(0.0, 1.0, len(seg_raw))
        cyc_raw = np.interp(x_new, x_old, seg_raw)
        cyc_bp  = np.interp(x_new, x_old, seg_bp)
        # Phase-shift: put the trough at index 0
        trough_idx = int(np.argmin(cyc_bp))
        cyc_raw = np.roll(cyc_raw, -trough_idx)
        cyc_bp  = np.roll(cyc_bp,  -trough_idx)
        raw_shapes.append(cyc_raw)
        bp_shapes.append(cyc_bp)
    if not raw_shapes:
        return np.empty((0, n_points)), np.empty((0, n_points))
    return np.stack(raw_shapes, 0), np.stack(bp_shapes, 0)


# ---- accumulate across all sessions for one rat ----
phasic_raw_all, phasic_bp_all = [], []
tonic_raw_all,  tonic_bp_all  = [], []
n_sessions_used = 0

for sess in list_rat_sessions(INSPECT_RAT):
    try:
        hpc, hypno, fs = get_data(sess['hpc_path'], sess['state_path'], type='hpc')
    except Exception:
        continue
    hpc = np.asarray(hpc, dtype=float)
    fs = int(fs)
    try:
        phasic_iv, tonic_iv, _ = extract_pt_intervals(hpc, hypno, fs=fs)
    except Exception:
        continue
    p_starts, p_ends = _intervalset_to_arrays(phasic_iv) if len(phasic_iv) else (np.array([]), np.array([]))
    t_starts, t_ends = _intervalset_to_arrays(tonic_iv)  if len(tonic_iv)  else (np.array([]), np.array([]))
    if len(p_starts) == 0 and len(t_starts) == 0:
        continue
    for starts, ends, raw_acc, bp_acc in [
        (p_starts, p_ends, phasic_raw_all, phasic_bp_all),
        (t_starts, t_ends, tonic_raw_all,  tonic_bp_all),
    ]:
        for s_sec, e_sec in zip(starts, ends):
            if e_sec - s_sec < MIN_INTERVAL_SEC:
                continue
            s_idx = int(round(s_sec * fs))
            e_idx = int(round(e_sec * fs))
            r, b = trough_aligned_cycle_shapes(hpc[s_idx:e_idx], fs)
            if r.size == 0:
                continue
            raw_acc.append(r); bp_acc.append(b)
    n_sessions_used += 1

phasic_raw = np.concatenate(phasic_raw_all, 0) if phasic_raw_all else np.empty((0, N_POINTS))
phasic_bp  = np.concatenate(phasic_bp_all,  0) if phasic_bp_all  else np.empty((0, N_POINTS))
tonic_raw  = np.concatenate(tonic_raw_all,  0) if tonic_raw_all  else np.empty((0, N_POINTS))
tonic_bp   = np.concatenate(tonic_bp_all,   0) if tonic_bp_all   else np.empty((0, N_POINTS))

print(f'sessions used: {n_sessions_used}')
print(f'phasic cycles: {len(phasic_raw)}')
print(f'tonic  cycles: {len(tonic_raw)}')

# ---- plot, x-axis goes 0 -> 360 deg, with TROUGH at 0 and 360 ----
x_deg = np.linspace(0.0, 360.0, N_POINTS, endpoint=False)
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5),
                          sharex=True, constrained_layout=True)
panels = [
    (axes[0, 0], phasic_raw, 'phasic — raw LFP',          'tab:blue'),
    (axes[0, 1], tonic_raw,  'tonic — raw LFP',           'tab:orange'),
    (axes[1, 0], phasic_bp,  'phasic — bandpass 5–12 Hz', 'tab:blue'),
    (axes[1, 1], tonic_bp,   'tonic — bandpass 5–12 Hz',  'tab:orange'),
]
for ax, data, label, color in panels:
    if data.size == 0:
        ax.set_title(f'{label}: no data'); ax.set_axis_off(); continue
    mean = data.mean(axis=0)
    sd   = data.std(axis=0, ddof=1)
    ax.plot(x_deg, mean, color=color, lw=2.0, label='mean')
    ax.fill_between(x_deg, mean - sd, mean + sd,
                    color=color, alpha=0.22, linewidth=0, label='±1 SD')
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0,   color='gray', lw=0.4, linestyle=':', alpha=0.7)
    ax.axvline(180, color='gray', lw=0.4, linestyle=':', alpha=0.7)
    ax.axvline(360, color='gray', lw=0.4, linestyle=':', alpha=0.7)
    ax.set_title(f'{label}  (n = {len(data)} cycles)')
    ax.set_ylabel('amplitude (a.u.)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right', frameon=False)
axes[-1, 0].set_xlabel('phase within cycle (deg)   — trough at 0/360, peak ~180')
axes[-1, 1].set_xlabel('phase within cycle (deg)   — trough at 0/360, peak ~180')

fig.suptitle(
    f'Rat {INSPECT_RAT} — average theta cycle shape, trough-aligned\n'
    f'each cycle resampled to {N_POINTS} pts, then rolled so bandpass-trough is at 0°. '
    f'shaded = ±1 SD',
    fontsize=11,
)
plt.show()

### Phasic cycle duration statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

rows = []
all_phasic_durs_ms = []   # pooled across rats, for the histogram

for group, rats in results.items():
    for rid, agg in rats.items():
        # half-open cycle bounds, sample indices at agg.fpps.fs (625 Hz default)
        bounds = agg.fpps.cycle_bounds                 # (n_cycles, 2)
        durs_sec = (bounds[:, 1] - bounds[:, 0]) / agg.fpps.fs
        durs_ms = durs_sec * 1000.0

        phasic = agg.cycle_substates == 'phasic'
        ph = durs_ms[phasic]
        all_phasic_durs_ms.append(ph)

        if ph.size:
            rows.append(dict(
                group=group, rat=rid,
                n_phasic=int(ph.size),
                dur_mean_ms=float(ph.mean()),
                dur_median_ms=float(np.median(ph)),
                dur_std_ms=float(ph.std(ddof=1)),
                dur_min_ms=float(ph.min()),
                dur_max_ms=float(ph.max()),
                dur_p5_ms=float(np.percentile(ph, 5)),
                dur_p95_ms=float(np.percentile(ph, 95)),
                inst_freq_mean_Hz=float(1000.0 / ph.mean()),
                inst_freq_median_Hz=float(1000.0 / np.median(ph)),
            ))

phasic_stats = pd.DataFrame(rows).sort_values(['group', 'rat']).reset_index(drop=True)
display(phasic_stats)

all_phasic = np.concatenate(all_phasic_durs_ms) if all_phasic_durs_ms else np.array([])

print()
print(f'pooled across all rats:  n_phasic_cycles = {len(all_phasic)}')
if all_phasic.size:
    print(f'  mean    = {all_phasic.mean():.1f} ms   ({1000.0/all_phasic.mean():.2f} Hz)')
    print(f'  median  = {np.median(all_phasic):.1f} ms')
    print(f'  std     = {all_phasic.std(ddof=1):.1f} ms')
    print(f'  min/max = {all_phasic.min():.1f} / {all_phasic.max():.1f} ms')
    print(f'  5%/95%  = {np.percentile(all_phasic,5):.1f} / {np.percentile(all_phasic,95):.1f} ms')

# Histogram + per-rat strip
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# (a) pooled histogram
ax = axes[0]
ax.hist(all_phasic, bins=60, color='tab:blue', alpha=0.7, edgecolor='black', linewidth=0.3)
for f_ref in (5.0, 8.0, 12.0):
    ax.axvline(1000.0 / f_ref, color='red', lw=0.7, linestyle='--', alpha=0.7,
                label=f'{f_ref:g} Hz = {1000.0/f_ref:.0f} ms')
ax.set_xlabel('phasic cycle duration (ms)')
ax.set_ylabel('# cycles')
ax.set_title(f'Pooled phasic cycle durations  (n = {len(all_phasic)})')
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)

# (b) per-rat boxplot
ax = axes[1]
data_per_rat = []
labels = []
for r in rows:
    g, rid = r['group'], r['rat']
    mask = (phasic_stats.group == g) & (phasic_stats.rat == rid)
    # fetch this rat's per-cycle durations
    for group, rats in results.items():
        if group != g: continue
        agg = rats[rid]
        d = (agg.fpps.cycle_bounds[:, 1] - agg.fpps.cycle_bounds[:, 0]) / agg.fpps.fs * 1000.0
        d = d[agg.cycle_substates == 'phasic']
        data_per_rat.append(d)
        labels.append(f'{g[:3]}.r{rid}')
        break
ax.boxplot(data_per_rat, tick_labels=labels, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue', alpha=0.6),
           medianprops=dict(color='black'))
for f_ref in (5.0, 8.0, 12.0):
    ax.axhline(1000.0 / f_ref, color='red', lw=0.6, linestyle='--', alpha=0.7)
ax.set_ylabel('phasic cycle duration (ms)')
ax.set_xlabel('rat')
ax.set_title('Per-rat phasic cycle duration')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)
plt.show()

### Total concatenated duration of phasic and tonic cycles

In [ ]:
import numpy as np
import pandas as pd

rows = []
for group, rats in results.items():
    for rid, agg in rats.items():
        bounds = agg.fpps.cycle_bounds                    # (n_cycles, 2) in samples
        durs_sec = (bounds[:, 1] - bounds[:, 0]) / agg.fpps.fs
        ph_mask = agg.cycle_substates == 'phasic'
        to_mask = agg.cycle_substates == 'tonic'
        rows.append(dict(
            group=group, rat=rid,
            n_phasic=int(ph_mask.sum()),
            phasic_total_sec=float(durs_sec[ph_mask].sum()),
            n_tonic=int(to_mask.sum()),
            tonic_total_sec=float(durs_sec[to_mask].sum()),
        ))

df = pd.DataFrame(rows).sort_values(['group', 'rat']).reset_index(drop=True)
df['phasic_total_min']  = df['phasic_total_sec'] / 60.0
df['tonic_total_min']   = df['tonic_total_sec']  / 60.0
display(df)

# Pooled totals
n_ph = int(df['n_phasic'].sum())
n_to = int(df['n_tonic'].sum())
tot_ph_sec = float(df['phasic_total_sec'].sum())
tot_to_sec = float(df['tonic_total_sec'].sum())

print()
print(f'POOLED across all rats:')
print(f'  phasic cycles: n = {n_ph:>7d}   total = '
      f'{tot_ph_sec:>9.1f} s  =  {tot_ph_sec/60:>6.2f} min  =  {tot_ph_sec/3600:>5.3f} h')
print(f'  tonic  cycles: n = {n_to:>7d}   total = '
      f'{tot_to_sec:>9.1f} s  =  {tot_to_sec/60:>6.2f} min  =  {tot_to_sec/3600:>5.3f} h')
print()
print(f'  ratio   tonic / phasic   (cycles)   = {n_to/n_ph:.2f}')
print(f'  ratio   tonic / phasic   (duration) = {tot_to_sec/tot_ph_sec:.2f}')
print(f'  effective phasic theta freq = {n_ph/tot_ph_sec:.2f} Hz')
print(f'  effective tonic  theta freq = {n_to/tot_to_sec:.2f} Hz')

## 2 — TG-state clustering

Per-rat clustering is the most faithful reflection of Zhang's approach
(channels are clustered individually, and the result is then summarised
across animals). We cluster each rat's full set of REM cycles
(phasic + tonic together) and label the four clusters as S, M, EF, LF.


In [ ]:
per_rat_cluster = {'positive': {}, 'control': {}}
for group, rats in results.items():
    for rid, agg in rats.items():
        if len(agg.fpps) < 200:
            print(f'rat {rid}: only {len(agg.fpps)} cycles, skipping cluster')
            continue
        cl = cluster_fpps_into_tg_states(
            agg.fpps.fpps,
            agg.frequencies,
            agg.phase_centers_rad,
            k=4, random_state=0, n_init=20,
        )
        per_rat_cluster[group][rid] = cl
        print(f'rat {rid}: gravity_freqs = '
              f'{np.round(cl.gravity_freqs,1)} Hz, '
              f'gravity_phases = {np.round(np.degrees(cl.gravity_phases),0)} deg')


### m-FPP panel per rat

Each row shows the four sorted m-FPPs for one rat (S, M, EF, LF). The
white contour traces the >95% peak field; the triangle marks the gravity
centre.


In [ ]:
FPP_CMAP = 'hot'   # m-FPP colormap (user request)

group_to_show = 'positive'
rats_to_show = list(per_rat_cluster[group_to_show].keys())
if not rats_to_show:
    print('no clusters in', group_to_show)
else:
    fig, axes = plt.subplots(len(rats_to_show), 4,
                             figsize=(13, 3.0*len(rats_to_show)),
                             constrained_layout=True)
    if len(rats_to_show) == 1:
        axes = axes[None, :]
    for ri, rid in enumerate(rats_to_show):
        cl = per_rat_cluster[group_to_show][rid]
        phase_deg = np.degrees(cl.phase_centers_rad)
        for s in range(4):
            zplot.plot_fpp(
                cl.m_fpps[s], cl.frequencies, phase_deg, ax=axes[ri, s],
                cmap=FPP_CMAP,
                title=f'rat {rid} - {STATE_NAMES[s]}\n'
                      f'{cl.gravity_freqs[s]:.1f} Hz, '
                      f'{np.degrees(cl.gravity_phases[s]):+.0f} deg',
                gravity_freq=cl.gravity_freqs[s],
                gravity_phase_deg=float(np.degrees(cl.gravity_phases[s]) % 360),
                mask=cl.masks[s],
            )
    plt.show()


### m-FPP per rat, phasic vs tonic separated

For each rat, the four sorted m-FPPs are recomputed twice: once using
only its **phasic** cycles, once using only its **tonic** cycles. Each
rat is one figure with 2 rows (phasic top, tonic bottom) × 4 columns
(S, M, EF, LF). Gravity centre + 95%-peak field mask are recomputed on
each substate-specific m-FPP. Triangles → substate-specific gravity
(white-fill = phasic, black-fill = tonic).

A within-rat colour scale is used so phasic and tonic are directly
comparable.


In [ ]:
from zhang_tg_states import (
    mean_fpp_per_cluster_substate, gravity_features,
)

group_to_show_pt = 'positive'
rats_to_show_pt = list(per_rat_cluster[group_to_show_pt].keys())

for rid in rats_to_show_pt:
    cl = per_rat_cluster[group_to_show_pt][rid]
    agg = results[group_to_show_pt][rid]
    m_by_sub = mean_fpp_per_cluster_substate(
        agg.fpps.fpps, cl.labels, agg.cycle_substates, n_clusters=4,
    )
    # Per-substate gravity / mask
    feats_phasic, feats_tonic = [], []
    for s in range(4):
        feats_phasic.append(gravity_features(
            m_by_sub['phasic'][s], cl.frequencies, cl.phase_centers_rad))
        feats_tonic.append(gravity_features(
            m_by_sub['tonic'][s], cl.frequencies, cl.phase_centers_rad))

    # Within-rat shared colour range across all 8 panels
    stack = np.concatenate([m_by_sub['phasic'].ravel(),
                            m_by_sub['tonic'].ravel()])
    vmin = float(np.nanpercentile(stack, 1))
    vmax = float(np.nanpercentile(stack, 99))

    fig, axes = plt.subplots(2, 4, figsize=(13, 6),
                             constrained_layout=True, sharex=True, sharey=True)
    phase_deg = np.degrees(cl.phase_centers_rad)
    n_ph = int((agg.cycle_substates == 'phasic').sum())
    n_to = int((agg.cycle_substates == 'tonic').sum())
    for s in range(4):
        # phasic
        zplot.plot_fpp(
            m_by_sub['phasic'][s], cl.frequencies, phase_deg,
            ax=axes[0, s], cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'rat {rid}  {STATE_NAMES[s]}  phasic\n'
                  f'{feats_phasic[s]["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats_phasic[s]["gravity_phase"]):+.0f} deg',
            gravity_freq=feats_phasic[s]['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats_phasic[s]['gravity_phase']) % 360),
            mask=feats_phasic[s]['mask'],
        )
        # tonic
        zplot.plot_fpp(
            m_by_sub['tonic'][s], cl.frequencies, phase_deg,
            ax=axes[1, s], cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'rat {rid}  {STATE_NAMES[s]}  tonic\n'
                  f'{feats_tonic[s]["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats_tonic[s]["gravity_phase"]):+.0f} deg',
            gravity_freq=feats_tonic[s]['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats_tonic[s]['gravity_phase']) % 360),
            mask=feats_tonic[s]['mask'],
        )
    fig.suptitle(f'rat {rid}  (n_phasic={n_ph}, n_tonic={n_to})', fontsize=11)
    plt.show()


### Group-average m-FPP (all REM cycles)

The four sorted m-FPPs are averaged across all rats in each group
(`positive` = RGS14, `control` = WT). One figure with 2 rows × 4
columns. Gravity centre + mask are recomputed on the group-averaged
m-FPP.


In [ ]:
# Build group-averaged m-FPPs (all cycles)
group_m_fpps = {}    # group -> (4, n_freq, n_phase)
group_n_rats = {}
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    group_n_rats[group] = len(rats)
    if not rats:
        group_m_fpps[group] = None
        continue
    stack = np.stack([per_rat_cluster[group][r].m_fpps for r in rats], axis=0)
    group_m_fpps[group] = np.nanmean(stack, axis=0)

# Shared colour range so positive and control are directly comparable
flat = np.concatenate([v.ravel() for v in group_m_fpps.values() if v is not None])
vmin = float(np.nanpercentile(flat, 1))
vmax = float(np.nanpercentile(flat, 99))

fig, axes = plt.subplots(2, 4, figsize=(13, 6),
                         constrained_layout=True, sharex=True, sharey=True)
freqs = FREQUENCIES
phase_deg = PHASE_CENTERS_DEG
for gi, group in enumerate(['positive', 'control']):
    gm = group_m_fpps[group]
    n_rats = group_n_rats[group]
    if gm is None:
        for s in range(4):
            axes[gi, s].set_axis_off()
        continue
    for s in range(4):
        feats = gravity_features(gm[s], freqs, PHASE_CENTERS_RAD)
        zplot.plot_fpp(
            gm[s], freqs, phase_deg, ax=axes[gi, s],
            cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'{group} (n_rats={n_rats})  {STATE_NAMES[s]}\n'
                  f'{feats["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats["gravity_phase"]):+.0f} deg',
            gravity_freq=feats['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats['gravity_phase']) % 360),
            mask=feats['mask'],
        )
plt.show()


### Group-average m-FPP, phasic vs tonic separated

For each group: m-FPPs are recomputed per state and per substate within
each rat, then averaged across rats. Result = 4 rows (positive-phasic,
positive-tonic, control-phasic, control-tonic) × 4 columns (S, M, EF,
LF). Shared colour scale across the whole figure.


In [ ]:
# Per-rat per-substate m-FPPs, then average across rats per group
group_m_by_sub = {}    # group -> {'phasic': (4,nf,np), 'tonic': (4,nf,np)}
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if not rats:
        group_m_by_sub[group] = None
        continue
    rat_ph = []
    rat_to = []
    for rid in rats:
        cl = per_rat_cluster[group][rid]
        agg = results[group][rid]
        m_by = mean_fpp_per_cluster_substate(
            agg.fpps.fpps, cl.labels, agg.cycle_substates, n_clusters=4,
        )
        rat_ph.append(m_by['phasic'])
        rat_to.append(m_by['tonic'])
    group_m_by_sub[group] = {
        'phasic': np.nanmean(np.stack(rat_ph, axis=0), axis=0),
        'tonic':  np.nanmean(np.stack(rat_to, axis=0), axis=0),
        'n_rats': len(rats),
    }

# Shared colour scale across the whole figure
all_arrays = []
for v in group_m_by_sub.values():
    if v is None:
        continue
    all_arrays.append(v['phasic'].ravel())
    all_arrays.append(v['tonic'].ravel())
flat = np.concatenate(all_arrays) if all_arrays else np.array([0.0, 1.0])
vmin = float(np.nanpercentile(flat, 1))
vmax = float(np.nanpercentile(flat, 99))

fig, axes = plt.subplots(4, 4, figsize=(13, 12),
                         constrained_layout=True, sharex=True, sharey=True)
row_layout = [
    ('positive', 'phasic'),
    ('positive', 'tonic'),
    ('control',  'phasic'),
    ('control',  'tonic'),
]
for ri, (group, substate) in enumerate(row_layout):
    block = group_m_by_sub.get(group)
    if block is None:
        for s in range(4):
            axes[ri, s].set_axis_off()
        continue
    arr = block[substate]
    n_rats = block['n_rats']
    for s in range(4):
        feats = gravity_features(arr[s], FREQUENCIES, PHASE_CENTERS_RAD)
        zplot.plot_fpp(
            arr[s], FREQUENCIES, PHASE_CENTERS_DEG, ax=axes[ri, s],
            cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'{group} {substate} (n={n_rats})\n'
                  f'{STATE_NAMES[s]}  {feats["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats["gravity_phase"]):+.0f} deg',
            gravity_freq=feats['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats['gravity_phase']) % 360),
            mask=feats['mask'],
        )
plt.show()


### Polar density of gravity centres

Aggregating across rats: each point is one rat's gravity centre for one
TG state. Angle = gravity phase, radius = gravity frequency.


In [ ]:
fig = plt.figure(figsize=(8, 4))
for gi, group in enumerate(['positive', 'control']):
    ax = fig.add_subplot(1, 2, gi+1, projection='polar')
    for s in range(4):
        freqs = [cl.gravity_freqs[s] for cl in per_rat_cluster[group].values()]
        phases = [cl.gravity_phases[s] for cl in per_rat_cluster[group].values()]
        if not freqs:
            continue
        ax.scatter(phases, freqs,
                   color=zplot.STATE_COLORS[s], label=STATE_NAMES[s],
                   s=80, alpha=0.7, edgecolor='black')
    ax.set_rlim(0, 180); ax.set_rticks([60,120,180])
    ax.set_title(f'{group}')
    if gi == 1:
        ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1.05), fontsize=8)
plt.suptitle('Gravity centres per rat (angle = phase, radius = freq)')
plt.show()


## 3 — Intra- vs inter-cluster correlation (5-fold CV)

For each cycle, intra = correlation with its own state's mean FPP;
inter_max = best correlation with any *other* state's mean FPP. Most
cycles should sit well above the inter line; cycles whose gap is near
zero share features with multiple states (Zhang reports ~20%).


In [ ]:
ii_per_rat = {'positive': {}, 'control': {}}
for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        ii_per_rat[group][rid] = kfold_intra_inter(
            agg.fpps.fpps, agg.frequencies, agg.phase_centers_rad,
            n_folds=5, k=4, random_state=0,
        )

# Plot intra and inter distributions across all rats
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for gi, group in enumerate(['positive', 'control']):
    intras = np.concatenate([r.intra for r in ii_per_rat[group].values()])
    inters = np.concatenate([r.inter_max for r in ii_per_rat[group].values()])
    axes[gi].hist(intras, bins=80, color='tab:blue', alpha=0.7,
                  label='intra-cluster')
    axes[gi].hist(inters, bins=80, color='tab:red', alpha=0.5,
                  label='max inter-cluster')
    axes[gi].set_xlabel('correlation r')
    axes[gi].set_ylabel('# cycles')
    axes[gi].set_title(f'{group}  (n={len(intras)})')
    axes[gi].legend()
plt.show()

# Gap distribution (intra - inter_max)
fig, ax = plt.subplots(figsize=(5, 3.2))
for group, color in [('positive','C0'), ('control','C1')]:
    gaps = np.concatenate([r.gap for r in ii_per_rat[group].values()])
    ax.hist(gaps, bins=80, alpha=0.55, color=color, label=group, density=True)
ax.axvline(0.0, color='k', lw=0.6, linestyle='--')
ax.axvline(0.05, color='gray', lw=0.6, linestyle=':')
ax.set_xlabel('intra - inter_max');  ax.set_ylabel('density')
ax.set_title('Cycle uniqueness to its assigned state')
ax.legend()
plt.show()


## 4 — Cross-rat assignment accuracy

Take rat $i$'s cycles, classify them with rat $j$'s reference m-FPPs.
Compare against rat $i$'s native cluster labels. Diagonal is 1.0 by
construction. Off-diagonal entries quantify how well a rat's TG state
definitions transfer.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if len(rats) < 2:
        continue
    fpps_list = [results[group][r].fpps.fpps for r in rats]
    labels_list = [per_rat_cluster[group][r].labels for r in rats]
    m_fpps_list = [per_rat_cluster[group][r].m_fpps for r in rats]
    acc = pairwise_cross_dataset(fpps_list, labels_list, m_fpps_list)
    fig, ax = plt.subplots(figsize=(4.0, 3.4))
    im = ax.imshow(acc, vmin=0.25, vmax=1.0, cmap='viridis')
    ax.set_xticks(range(len(rats))); ax.set_yticks(range(len(rats)))
    ax.set_xticklabels([f'r{r}' for r in rats], rotation=0)
    ax.set_yticklabels([f'r{r}' for r in rats])
    ax.set_title(f'cross-rat accuracy ({group})')
    for i in range(len(rats)):
        for j in range(len(rats)):
            ax.text(j, i, f'{acc[i,j]:.2f}', ha='center', va='center',
                    fontsize=7,
                    color='white' if acc[i,j] < 0.6 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.show()
    off = acc[~np.eye(len(rats), dtype=bool)]
    print(f'{group}: off-diagonal accuracy = {off.mean():.2f} '
          f'(min {off.min():.2f}, max {off.max():.2f}, chance = 0.25)')


## 5 — Markov state transitions: phasic vs tonic

For each rat we tag every cycle as phasic, tonic, or excluded, build the
state sequence within each contiguous phasic / tonic interval, and
compute a 4×4 transition matrix per substate. Transitions across
substate boundaries are not counted.


In [ ]:
from zhang_tg_states import transitions_by_substate

trans_per_rat = {'positive': {}, 'control': {}}

for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        # Each agg.cycle_segment_ids[i] is a unique integer for the
        # phasic/tonic interval the cycle belongs to. Runs = groups of
        # cycles sharing a segment_id (= cycles within ONE REM substate
        # interval). This avoids reconstructing interval bounds from the
        # non-chronological concatenated cycle array.
        tt = transitions_by_substate(
            labels=cl.labels,
            substates=agg.cycle_substates,
            segment_ids=agg.cycle_segment_ids,
        )
        trans_per_rat[group][rid] = tt

# Plot per-group averaged transition matrices and occurrence bars
for group in ['positive', 'control']:
    rats = list(trans_per_rat[group].keys())
    if not rats:
        continue
    Tph = np.mean([trans_per_rat[group][r]['phasic'].transition_matrix for r in rats], axis=0)
    Tto = np.mean([trans_per_rat[group][r]['tonic'].transition_matrix for r in rats], axis=0)
    Oph = np.mean([trans_per_rat[group][r]['phasic'].occurrence for r in rats], axis=0)
    Oto = np.mean([trans_per_rat[group][r]['tonic'].occurrence for r in rats], axis=0)

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), constrained_layout=True)
    zplot.plot_transition_matrix(Tph, ax=axes[0], title=f'{group} phasic (n_rats={len(rats)})')
    zplot.plot_transition_matrix(Tto, ax=axes[1], title=f'{group} tonic (n_rats={len(rats)})')
    zplot.plot_transition_diff(Tph, Tto, ax=axes[2], title='phasic - tonic')
    zplot.plot_state_occurrence({'phasic': Oph, 'tonic': Oto},
                                ax=axes[3], title=f'occurrence ({group})')
    plt.show()


## 6 — PFC-HPC PPC per TG state, phasic vs tonic

For each rat:

* split cycles by `substate` ∈ {phasic, tonic};
* within each subset, split by TG state {S, M, EF, LF};
* compute PPC across the cycles for each (substate × state) cell, then
  average PPC heatmaps across rats for the group-level plot.

Convention: `phase_lag = angle(PFC * conj(HPC))`. Higher PPC = more
reliable PFC→HPC phase lag at that (frequency, theta phase).


In [ ]:
def state_ppc_split(agg, labels):
    out = {}
    for substate in ['phasic', 'tonic']:
        mask = agg.cycle_substates == substate
        if not np.any(mask):
            out[substate] = None
            continue
        res = state_conditioned_ppc(agg.cross_spectrum.angles[mask],
                                    labels[mask],
                                    frequencies=agg.frequencies,
                                    phase_centers_deg=PHASE_CENTERS_DEG)
        out[substate] = res
    return out

per_rat_ppc = {'positive': {}, 'control': {}}
for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        per_rat_ppc[group][rid] = state_ppc_split(agg, cl.labels)

# Group means
def stack_group(group):
    arr_phasic = []
    arr_tonic = []
    nph = []
    nto = []
    for rid in per_rat_ppc[group]:
        ph = per_rat_ppc[group][rid]['phasic']
        to = per_rat_ppc[group][rid]['tonic']
        if ph is None or to is None:
            continue
        arr_phasic.append(ph.ppc_per_state)
        arr_tonic.append(to.ppc_per_state)
        nph.append(ph.n_cycles_per_state)
        nto.append(to.n_cycles_per_state)
    if not arr_phasic:
        return None, None, None, None
    return (np.nanmean(np.stack(arr_phasic, 0), axis=0),
            np.nanmean(np.stack(arr_tonic, 0), axis=0),
            np.sum(np.stack(nph, 0), axis=0),
            np.sum(np.stack(nto, 0), axis=0))

for group in ['positive', 'control']:
    g_ph, g_to, n_ph, n_to = stack_group(group)
    if g_ph is None:
        continue
    print(f'group {group}: cycles per state — phasic {n_ph}, tonic {n_to}')
    vmax = float(np.nanpercentile(np.concatenate([g_ph.ravel(), g_to.ravel()]), 99))
    fig = zplot.plot_phasic_vs_tonic_ppc(
        g_ph, g_to, FREQUENCIES, PHASE_CENTERS_DEG,
        vmin=0.0, vmax=vmax, title_prefix=f'PFC-HPC PPC — {group}',
    )
    plt.show()


### PPC(f) curves averaged in the gravity phase window

For each TG state, the PPC heatmap is summarised by averaging across the
gravity-centred phase window `[phase − 7σ, phase + σ]` (Zhang). The
result is a single PPC vs frequency curve per state.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if not rats:
        continue

    # Compute PPC(f) per rat using THAT rat's own gravity phase and std
    # so the inter-rat std reflects honest variability (cluster boundaries
    # and PPC heatmap both jitter across animals).
    n_states = 4
    per_rat_ppc_f_phasic = []   # each entry: (n_states, n_freq)
    per_rat_ppc_f_tonic = []
    rat_labels = []
    for rid in rats:
        cl = per_rat_cluster[group][rid]
        pr = per_rat_ppc[group][rid]
        if pr['phasic'] is None or pr['tonic'] is None:
            continue
        ph = pr['phasic'].ppc_per_state    # (4, n_freq, n_phase)
        to = pr['tonic'].ppc_per_state
        gp = cl.gravity_phases             # (4,)
        psd = cl.phase_stds                # (4,)
        ppc_f_ph = np.zeros((n_states, len(FREQUENCIES)))
        ppc_f_to = np.zeros((n_states, len(FREQUENCIES)))
        for s in range(n_states):
            ppc_f_ph[s], _ = ppc_phase_pooled(ph[s], gp[s], psd[s],
                                              phase_centers_rad=PHASE_CENTERS_RAD)
            ppc_f_to[s], _ = ppc_phase_pooled(to[s], gp[s], psd[s],
                                              phase_centers_rad=PHASE_CENTERS_RAD)
        per_rat_ppc_f_phasic.append(ppc_f_ph)
        per_rat_ppc_f_tonic.append(ppc_f_to)
        rat_labels.append(rid)
    if not per_rat_ppc_f_phasic:
        continue

    stack_ph = np.stack(per_rat_ppc_f_phasic, axis=0)  # (n_rats, n_states, n_freq)
    stack_to = np.stack(per_rat_ppc_f_tonic, axis=0)
    mean_ph = np.nanmean(stack_ph, axis=0)
    std_ph  = np.nanstd(stack_ph, axis=0, ddof=1) if len(rat_labels) > 1 else np.zeros_like(mean_ph)
    mean_to = np.nanmean(stack_to, axis=0)
    std_to  = np.nanstd(stack_to, axis=0, ddof=1) if len(rat_labels) > 1 else np.zeros_like(mean_to)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True,
                             constrained_layout=True)
    zplot.plot_ppc_per_freq(mean_ph, FREQUENCIES,
                            ax=axes[0],
                            title=f'{group} - phasic  (n_rats = {len(rat_labels)})',
                            ppc_per_state_freq_std=std_ph)
    zplot.plot_ppc_per_freq(mean_to, FREQUENCIES,
                            ax=axes[1],
                            title=f'{group} - tonic  (n_rats = {len(rat_labels)})',
                            ppc_per_state_freq_std=std_to)
    axes[0].set_ylabel('PPC')
    plt.show()


## 7 — Persist the artifacts

Save the per-rat cluster results, validation results, transition matrices,
and PPC matrices to disk for downstream analysis.


In [ ]:
from zhang_tg_states import mean_fpp_per_cluster_substate

out_dir = os.path.join(HERE, 'zhang_tg_states')
os.makedirs(out_dir, exist_ok=True)

def to_dict(cl, agg=None):
    out = dict(
        labels=cl.labels, m_fpps=cl.m_fpps,
        gravity_freqs=cl.gravity_freqs, gravity_phases=cl.gravity_phases,
        freq_stds=cl.freq_stds, phase_stds=cl.phase_stds,
        masks=cl.masks, raw_labels=cl.raw_labels, permutation=cl.permutation,
        frequencies=cl.frequencies, phase_centers_rad=cl.phase_centers_rad,
    )
    # also save per-substate m-FPPs so the "FPP x transitions" analyses
    # (section 10) work after a load-from-pickle without raw cycles.
    if agg is not None:
        try:
            m_by = mean_fpp_per_cluster_substate(
                agg.fpps.fpps, cl.labels, agg.cycle_substates, n_clusters=4,
            )
            out['m_fpps_phasic'] = m_by['phasic']
            out['m_fpps_tonic']  = m_by['tonic']
        except Exception:
            pass
    return out

def ppc_to_dict(d):
    return {k: (None if v is None else dict(
        ppc_per_state=v.ppc_per_state,
        n_cycles_per_state=v.n_cycles_per_state,
        n_per_freq_phase=v.n_per_freq_phase,
        frequencies=v.frequencies,
        phase_centers_deg=v.phase_centers_deg,
    )) for k, v in d.items()}

def trans_to_dict(d):
    return {k: dict(counts=v.counts,
                    transition_matrix=v.transition_matrix,
                    occurrence=v.occurrence,
                    n_cycles=v.n_cycles,
                    n_transitions=v.n_transitions) for k, v in d.items()}

for group in ['positive', 'control']:
    payload = dict(
        group=group,
        rat_ids=list(per_rat_cluster[group].keys()),
        clusters={r: to_dict(c, results.get(group, {}).get(r) if results is not None else None)
                   for r, c in per_rat_cluster[group].items()},
        intra_inter={r: dict(intra=v.intra, inter_max=v.inter_max,
                              labels=v.labels, gap=v.gap)
                     for r, v in ii_per_rat[group].items()},
        transitions={r: trans_to_dict(v)
                     for r, v in trans_per_rat[group].items()},
        ppc={r: ppc_to_dict(v) for r, v in per_rat_ppc[group].items()},
        cycle_summary=cycle_summary.to_dict(orient='records'),
        meta=dict(
            frequencies=FREQUENCIES, phase_centers_deg=PHASE_CENTERS_DEG,
            phase_centers_rad=PHASE_CENTERS_RAD, analysis_fs=ANALYSIS_FS,
            state_names=STATE_NAMES,
            source='PFC', target='HPC',
        ),
    )
    fname = os.path.join(out_dir, f'tg_states_v2_{group}_pfc_hpc.pkl')
    with open(fname, 'wb') as fh:
        pickle.dump(payload, fh)
    print('wrote', fname)


## 8 — Distribution of PPC values: phasic vs tonic

How spread out are the PPC values themselves across `(frequency, theta-phase)`
bins? If phasic REM produces *irregular* coupling — only a few hot spots
in the heatmap — then the phasic PPC distribution should have a wide
spread (some near-zero, some very high). If tonic REM produces a more
*regular*, uniform coupling, the tonic distribution should be tighter
around its mean.

For each group and each TG state we pool every PPC value across
`(rats × frequencies × phase-bins)` and plot the **density** for phasic
vs tonic side by side. Dashed verticals mark each substate's mean. The
title quotes mean (μ), std (σ), and IQR — a tighter σ means a more
regular coupling pattern; a long right tail means a few very
strongly-locked bins dominate the heatmap.

The right-most panel ('all states') pools across the four TG states so
you can see the global substate effect on coupling regularity.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_ppc[group].keys())
    if not rats:
        continue

    # pooled[substate][state] = list of (n_freq * n_phase,) arrays, one per rat
    pooled = {sub: {s: [] for s in range(4)} for sub in ('phasic', 'tonic')}
    for rid in rats:
        pr = per_rat_ppc[group][rid]
        for substate in ('phasic', 'tonic'):
            if pr[substate] is None:
                continue
            for s in range(4):
                pooled[substate][s].append(pr[substate].ppc_per_state[s].ravel())

    flat = {sub: {s: (np.concatenate(pooled[sub][s])
                       if pooled[sub][s] else np.array([]))
                  for s in range(4)}
            for sub in ('phasic', 'tonic')}
    for sub in ('phasic', 'tonic'):
        flat[sub]['all'] = (np.concatenate([flat[sub][s] for s in range(4)])
                            if any(flat[sub][s].size for s in range(4))
                            else np.array([]))

    cols = list(range(4)) + ['all']
    titles = list(STATE_NAMES) + ['all states pooled']

    fig, axes = plt.subplots(1, 5, figsize=(17, 3.6),
                             sharey=False, constrained_layout=True)
    for ci, (col, title) in enumerate(zip(cols, titles)):
        ph = flat['phasic'][col]; ph = ph[np.isfinite(ph)]
        to = flat['tonic'][col];  to = to[np.isfinite(to)]
        if ph.size == 0 or to.size == 0:
            axes[ci].set_axis_off()
            continue
        all_vals = np.concatenate([ph, to])
        vmin, vmax = np.nanpercentile(all_vals, [0.5, 99.5])
        bins = np.linspace(vmin, vmax, 60)
        axes[ci].hist(ph, bins=bins, color='tab:blue', alpha=0.55,
                      density=True, label='phasic')
        axes[ci].hist(to, bins=bins, color='tab:orange', alpha=0.55,
                      density=True, label='tonic')
        axes[ci].axvline(np.mean(ph), color='tab:blue',
                          linestyle='--', lw=1.0)
        axes[ci].axvline(np.mean(to), color='tab:orange',
                          linestyle='--', lw=1.0)
        iqr_ph = np.subtract(*np.percentile(ph, [75, 25]))
        iqr_to = np.subtract(*np.percentile(to, [75, 25]))
        axes[ci].set_title(
            f'{title}\n'
            f'phasic: μ={np.mean(ph):.3f}, σ={np.std(ph):.3f}, IQR={iqr_ph:.3f}\n'
            f'tonic:  μ={np.mean(to):.3f}, σ={np.std(to):.3f}, IQR={iqr_to:.3f}',
            fontsize=8,
        )
        axes[ci].set_xlabel('PPC')
        if ci == 0:
            axes[ci].set_ylabel('density')
        if ci == len(cols) - 1:
            axes[ci].legend(fontsize=8, frameon=False)
    fig.suptitle(f'PPC value distribution — {group}'
                 f'  (n_rats = {len(rats)})', fontsize=12)
    plt.show()


### Interpreting the σ and IQR

* **σ (std) and IQR small** → PPC values across the heatmap cluster
  tightly around the mean. Coupling is *regular*: most (f, θ) bins have
  similar PPC.
* **σ and IQR large**, or a heavy right tail in the histogram →
  coupling is *irregular*: a few bins are strongly locked while most
  are near zero.

Differences in mean answer "is coupling stronger overall?"; differences
in σ/IQR answer "is coupling more concentrated in specific (f, θ) bins,
or more uniform?". The two are independent — phasic could have the
same mean as tonic but a fatter tail, or vice versa.


## 10 — Substate and group comparisons

Direct contrasts of the two main quantities we have measured for each
state, with proper hypothesis tests across rats.

**10.1 — Transition probabilities: phasic vs tonic, per group.**
Per-cell paired t-test across rats (same rats, two substates).

**10.2 — Transition probabilities: positive vs control, per substate.**
Per-cell Welch t-test across rats (different rats, between groups).

**10.3 — Gamma power (mean m-FPP inside gravity field): phasic vs tonic.**
Per-state paired t-test across rats.

**10.4 — Gamma power: positive vs control, per substate.**
Per-state Welch t-test across rats.

P-values are FDR-corrected (Benjamini-Hochberg) for multiple comparisons
where shown with an asterisk. With only 4 rats per group, effect sizes
are more informative than p-values; always interpret the matched
bars/matrices alongside the test.


### 10.1 — Transition probabilities: phasic vs tonic, per group

In [ ]:
from scipy.stats import ttest_rel, ttest_ind
from statsmodels.stats.multitest import multipletests

def collect_T(group, substate):
    '''(n_rats, 4, 4) array of per-rat transition matrices.'''
    return np.stack([tt[substate].transition_matrix
                     for tt in trans_per_rat[group].values()], axis=0)

fig, axes = plt.subplots(2, 3, figsize=(13.5, 8),
                          constrained_layout=True)
for gi, group in enumerate(["positive", "control"]):
    T_ph = collect_T(group, "phasic")    # (n_rats, 4, 4)
    T_to = collect_T(group, "tonic")
    if T_ph.shape[0] < 2:
        for k in range(3):
            axes[gi, k].set_axis_off()
        continue
    diff = (T_ph - T_to).mean(axis=0)
    p_mat = np.full((4, 4), np.nan)
    for i in range(4):
        for j in range(4):
            t, p = ttest_rel(T_ph[:, i, j], T_to[:, i, j])
            p_mat[i, j] = p
    flat = p_mat.flatten()
    valid = ~np.isnan(flat)
    reject = np.zeros_like(flat, dtype=bool)
    if valid.any():
        rej, _, _, _ = multipletests(flat[valid], method="fdr_bh")
        reject[valid] = rej
    reject_mat = reject.reshape(4, 4)
    vlim = float(np.nanmax(np.abs(diff))) or 1e-3

    ax = axes[gi, 0]
    im = ax.imshow(diff, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{diff[i, j]:+.2f}", ha="center", va="center",
                    fontsize=7,
                    color="white" if abs(diff[i, j]) > 0.6 * vlim else "black")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(STATE_NAMES, rotation=30, fontsize=8)
    ax.set_yticklabels(STATE_NAMES, fontsize=8)
    ax.set_xlabel("next state"); ax.set_ylabel("current state")
    ax.set_title(f"{group}:  T_phasic - T_tonic  (mean over {T_ph.shape[0]} rats)")
    plt.colorbar(im, ax=ax, shrink=0.7)

    ax = axes[gi, 1]
    log_p = -np.log10(np.clip(p_mat, 1e-10, 1.0))
    im = ax.imshow(log_p, cmap="viridis", vmin=0, vmax=3)
    for i in range(4):
        for j in range(4):
            txt = f"{p_mat[i, j]:.3f}"
            if reject_mat[i, j]:
                txt += "*"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="white" if log_p[i, j] > 1.5 else "black")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(STATE_NAMES, rotation=30, fontsize=8)
    ax.set_yticklabels(STATE_NAMES, fontsize=8)
    ax.set_title(f"{group}:  paired t-test p-values\\n* = FDR significant")
    plt.colorbar(im, ax=ax, shrink=0.7, label="-log10(p)")

    O_ph = np.mean([tt["phasic"].occurrence for tt in trans_per_rat[group].values()],
                   axis=0)
    O_to = np.mean([tt["tonic"].occurrence for tt in trans_per_rat[group].values()],
                   axis=0)
    O_ph_sem = (np.std([tt["phasic"].occurrence for tt in trans_per_rat[group].values()],
                       axis=0, ddof=1) / np.sqrt(T_ph.shape[0]))
    O_to_sem = (np.std([tt["tonic"].occurrence for tt in trans_per_rat[group].values()],
                       axis=0, ddof=1) / np.sqrt(T_to.shape[0]))
    ax = axes[gi, 2]
    x = np.arange(4)
    ax.bar(x - 0.2, O_ph, yerr=O_ph_sem, width=0.4,
           label="phasic", color="tab:blue", alpha=0.7, edgecolor="black",
           linewidth=0.4)
    ax.bar(x + 0.2, O_to, yerr=O_to_sem, width=0.4,
           label="tonic", color="tab:orange", alpha=0.7, edgecolor="black",
           linewidth=0.4)
    ax.set_xticks(x); ax.set_xticklabels(STATE_NAMES, rotation=20, fontsize=8)
    ax.set_ylabel("state occurrence")
    ax.set_title(f"{group}: state occurrence")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle("10.1 — Transition probabilities: phasic vs tonic, per group",
             fontsize=12)
plt.show()


### 10.2 — Transition probabilities: positive vs control, per substate

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.5),
                          constrained_layout=True)
for si, substate in enumerate(["phasic", "tonic"]):
    T_pos = collect_T("positive", substate)
    T_ctl = collect_T("control",  substate)
    if T_pos.shape[0] < 2 or T_ctl.shape[0] < 2:
        for k in range(2):
            axes[si, k].set_axis_off()
        continue
    diff = T_pos.mean(0) - T_ctl.mean(0)
    p_mat = np.full((4, 4), np.nan)
    for i in range(4):
        for j in range(4):
            t, p = ttest_ind(T_pos[:, i, j], T_ctl[:, i, j], equal_var=False)
            p_mat[i, j] = p
    flat = p_mat.flatten()
    valid = ~np.isnan(flat)
    reject = np.zeros_like(flat, dtype=bool)
    if valid.any():
        rej, _, _, _ = multipletests(flat[valid], method="fdr_bh")
        reject[valid] = rej
    reject_mat = reject.reshape(4, 4)
    vlim = float(np.nanmax(np.abs(diff))) or 1e-3

    ax = axes[si, 0]
    im = ax.imshow(diff, cmap="RdBu_r", vmin=-vlim, vmax=vlim)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{diff[i, j]:+.2f}", ha="center", va="center",
                    fontsize=7,
                    color="white" if abs(diff[i, j]) > 0.6 * vlim else "black")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(STATE_NAMES, rotation=30, fontsize=8)
    ax.set_yticklabels(STATE_NAMES, fontsize=8)
    ax.set_xlabel("next state"); ax.set_ylabel("current state")
    ax.set_title(f"{substate}:  T_positive - T_control")
    plt.colorbar(im, ax=ax, shrink=0.7)

    ax = axes[si, 1]
    log_p = -np.log10(np.clip(p_mat, 1e-10, 1.0))
    im = ax.imshow(log_p, cmap="viridis", vmin=0, vmax=3)
    for i in range(4):
        for j in range(4):
            txt = f"{p_mat[i, j]:.3f}"
            if reject_mat[i, j]:
                txt += "*"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="white" if log_p[i, j] > 1.5 else "black")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(STATE_NAMES, rotation=30, fontsize=8)
    ax.set_yticklabels(STATE_NAMES, fontsize=8)
    ax.set_title(f"{substate}:  Welch t-test p-values\\n* = FDR significant")
    plt.colorbar(im, ax=ax, shrink=0.7, label="-log10(p)")

fig.suptitle("10.2 — Transition probabilities: positive vs control, per substate",
             fontsize=12)
plt.show()


### 10.3 — Gamma power inside gravity field: phasic vs tonic, per group

Gamma strength per rat per state = mean substate-specific m-FPP power
inside the substate-specific gravity field (`m_fpps_phasic[s]` >= 0.95 *
peak for the phasic value; same for tonic). Within each group we run a
paired t-test across rats. Lines connect the same rat across substates.


In [ ]:
from zhang_tg_states import mean_fpp_per_cluster_substate

def per_rat_gamma_strength(group, substate):
    '''Per-rat mean m-FPP power inside the substate-specific gamma field.
    Returns (n_rats, 4) and the ordered rat-id list.
    Looks for cached m_fpps_<substate> on each cluster first; if missing
    (i.e. you ran the notebook fresh rather than loaded from pickle),
    compute it from raw cycles in `results` and cache for next time.
    '''
    rows, rids = [], []
    for rid, cl in per_rat_cluster[group].items():
        m_sub = getattr(cl, f"m_fpps_{substate}", None)
        if m_sub is None:
            # fallback: compute on the fly from raw cycles
            if (results is not None
                    and group in results
                    and rid in results[group]):
                agg = results[group][rid]
                m_by = mean_fpp_per_cluster_substate(
                    agg.fpps.fpps, cl.labels, agg.cycle_substates,
                    n_clusters=4,
                )
                m_sub = m_by[substate]
                # cache for future calls / 10.4
                try:
                    setattr(cl, f"m_fpps_{substate}", m_sub)
                except Exception:
                    pass
        if m_sub is None:
            continue
        per_state = np.full(4, np.nan)
        for s in range(4):
            peak = np.nanmax(m_sub[s])
            if not np.isfinite(peak) or peak <= 0:
                continue
            mask = m_sub[s] >= 0.95 * peak
            per_state[s] = float(np.nanmean(m_sub[s][mask]))
        rows.append(per_state); rids.append(rid)
    return (np.stack(rows, 0) if rows else np.empty((0, 4))), rids


fig, axes = plt.subplots(2, 4, figsize=(14, 7),
                          constrained_layout=True, sharey="row")
gp_records = []
for gi, group in enumerate(["positive", "control"]):
    ph_vals, rids_ph = per_rat_gamma_strength(group, "phasic")
    to_vals, rids_to = per_rat_gamma_strength(group, "tonic")
    common = sorted(set(rids_ph) & set(rids_to))
    if not common:
        for s in range(4):
            axes[gi, s].set_axis_off()
        continue
    ph = ph_vals[[rids_ph.index(r) for r in common]]
    to = to_vals[[rids_to.index(r) for r in common]]
    for s in range(4):
        ax = axes[gi, s]
        m_ph, m_to = ph[:, s].mean(), to[:, s].mean()
        s_ph = ph[:, s].std(ddof=1) / np.sqrt(len(common)) if len(common) > 1 else 0
        s_to = to[:, s].std(ddof=1) / np.sqrt(len(common)) if len(common) > 1 else 0
        ax.bar([0, 1], [m_ph, m_to], yerr=[s_ph, s_to],
               width=0.55, color=["tab:blue", "tab:orange"],
               alpha=0.55, edgecolor="black", linewidth=0.6, capsize=4)
        for i in range(len(common)):
            ax.plot([0, 1], [ph[i, s], to[i, s]], "o-",
                    color="gray", alpha=0.7, markersize=6,
                    markeredgecolor="black", markeredgewidth=0.6)
        if len(common) >= 2:
            t, p = ttest_rel(ph[:, s], to[:, s])
            sig = " *" if p < 0.05 else ""
            ax.set_title(f"{group}  {STATE_NAMES[s]}\n"
                         f"paired-t p = {p:.3f}{sig}", fontsize=9)
            gp_records.append(dict(
                group=group, state=STATE_NAMES[s],
                phasic_mean=float(m_ph), tonic_mean=float(m_to),
                phasic_minus_tonic=float(m_ph - m_to),
                t=float(t), p=float(p), n_rats=len(common),
            ))
        else:
            ax.set_title(f"{group}  {STATE_NAMES[s]}", fontsize=9)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["phasic", "tonic"])
        if s == 0:
            ax.set_ylabel(f"gamma strength ({group})")
        ax.grid(alpha=0.3)

fig.suptitle("10.3 — Gamma power (mean m-FPP in gravity field): "
             "phasic vs tonic per group  (paired t-test, n = rats)",
             fontsize=12)
plt.show()

gamma_phasic_tonic_df = pd.DataFrame(gp_records)
display(gamma_phasic_tonic_df)

### 10.4 — Gamma power inside gravity field: positive vs control, per substate

In [ ]:
rng_jitter = np.random.default_rng(0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7),
                          constrained_layout=True, sharey="row")
gp_g_records = []
for si, substate in enumerate(["phasic", "tonic"]):
    pos_vals, rids_pos = per_rat_gamma_strength("positive", substate)
    ctl_vals, rids_ctl = per_rat_gamma_strength("control",  substate)
    for s in range(4):
        ax = axes[si, s]
        n_pos, n_ctl = len(pos_vals), len(ctl_vals)
        if n_pos == 0 or n_ctl == 0:
            ax.set_axis_off()
            continue
        m_pos = float(np.nanmean(pos_vals[:, s]))
        m_ctl = float(np.nanmean(ctl_vals[:, s]))
        sem_pos = float(np.nanstd(pos_vals[:, s], ddof=1) / np.sqrt(n_pos)) if n_pos > 1 else 0.0
        sem_ctl = float(np.nanstd(ctl_vals[:, s], ddof=1) / np.sqrt(n_ctl)) if n_ctl > 1 else 0.0
        ax.bar([0, 1], [m_pos, m_ctl], yerr=[sem_pos, sem_ctl],
               width=0.55,
               color=["tab:blue", "tab:orange"], alpha=0.55,
               edgecolor="black", linewidth=0.6, capsize=4)
        ax.scatter(np.zeros(n_pos) + 0.08 * rng_jitter.standard_normal(n_pos),
                    pos_vals[:, s], color="tab:blue", s=48, alpha=0.85,
                    edgecolor="black", linewidth=0.5)
        ax.scatter(np.ones(n_ctl) + 0.08 * rng_jitter.standard_normal(n_ctl),
                    ctl_vals[:, s], color="tab:orange", s=48, alpha=0.85,
                    edgecolor="black", linewidth=0.5)
        if n_pos >= 2 and n_ctl >= 2:
            t, p = ttest_ind(pos_vals[:, s], ctl_vals[:, s], equal_var=False)
            sig = " *" if p < 0.05 else ""
            ax.set_title(f"{substate}  {STATE_NAMES[s]}\n"
                         f"Welch p = {p:.3f}{sig}", fontsize=9)
            gp_g_records.append(dict(
                substate=substate, state=STATE_NAMES[s],
                positive_mean=m_pos, control_mean=m_ctl,
                positive_minus_control=m_pos - m_ctl,
                t=float(t), p=float(p),
                n_positive=n_pos, n_control=n_ctl,
            ))
        else:
            ax.set_title(f"{substate}  {STATE_NAMES[s]}", fontsize=9)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["positive", "control"])
        if s == 0:
            ax.set_ylabel(f"gamma strength ({substate})")
        ax.grid(alpha=0.3)

fig.suptitle("10.4 — Gamma power (mean m-FPP in gravity field): "
             "positive vs control per substate  (Welch t-test)",
             fontsize=12)
plt.show()

gamma_group_df = pd.DataFrame(gp_g_records)
display(gamma_group_df)

## Interpretation summary

* **Wavelet FPPs** capture the joint distribution of gamma power and
  theta phase per individual cycle, exactly as in Zhang Fig. 1B.
* The four TG states sort by gravity frequency (S < M < EF ≈ LF) and
  phase (S, M near θ peak; EF early descending; LF late descending).
* **Intra/inter correlation under 5-fold CV** quantifies how "pure"
  each cycle is. A near-zero gap means a cycle has features of more
  than one TG state.
* **Cross-rat accuracy** shows whether m-FPPs transfer between rats —
  consistently >> 0.25 (chance) means the four states are robust across
  animals.
* **Markov transitions** reveal whether the network *switches* states
  rapidly or persists, separately for phasic vs tonic REM. The
  occurrence bars and the (phasic − tonic) difference matrix highlight
  REM-substate-specific reorganisations.
* **PFC-HPC PPC** per TG state shows whether the inter-area phase lag
  is reliable in each oscillatory regime. Differences between phasic
  and tonic suggest substate-specific PFC↔HPC coordination — the
  question this notebook is built to answer.

This pipeline is a faithful adaptation of Zhang et al. 2019 with two
substantive substitutions: PFC for CA3/EC, and phasic/tonic REM for
pre/track/post.
